# Plugin

> Demucs v4 audio source separation plugin — provides vocals extraction for removing background noise and music from speech audio.

In [ ]:
#| default_exp plugin

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
import logging
import uuid
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional

import torch

from cjm_media_plugin_system.processing_interface import MediaProcessingPlugin
from cjm_media_plugin_system.core import MediaMetadata
from cjm_media_plugin_system.storage import MediaProcessingStorage

from cjm_plugin_system.utils.hashing import hash_file
from cjm_plugin_system.utils.validation import (
    dict_to_config, config_to_dict, dataclass_to_jsonschema,
    SCHEMA_TITLE, SCHEMA_DESC, SCHEMA_ENUM, SCHEMA_MIN, SCHEMA_MAX
)

## Configuration

In [ ]:
#| export
@dataclass
class DemucsPluginConfig:
    """Configuration for the Demucs processing plugin."""
    
    model: str = field(
        default="htdemucs",
        metadata={
            SCHEMA_TITLE: "Model",
            SCHEMA_DESC: "Demucs model to use for separation.",
            SCHEMA_ENUM: ["htdemucs", "htdemucs_ft", "htdemucs_6s", "mdx_extra", "mdx_extra_q"]
        }
    )
    
    device: str = field(
        default="auto",
        metadata={
            SCHEMA_TITLE: "Device",
            SCHEMA_DESC: "Compute device. 'auto' selects CUDA if available.",
            SCHEMA_ENUM: ["auto", "cpu", "cuda"]
        }
    )
    
    shifts: int = field(
        default=1,
        metadata={
            SCHEMA_TITLE: "Shifts",
            SCHEMA_DESC: "Number of random shifts for prediction averaging. Higher = better quality but slower.",
            SCHEMA_MIN: 0, SCHEMA_MAX: 10
        }
    )
    
    overlap: float = field(
        default=0.25,
        metadata={
            SCHEMA_TITLE: "Overlap",
            SCHEMA_DESC: "Overlap between prediction windows (0.0 to 0.5).",
            SCHEMA_MIN: 0.0, SCHEMA_MAX: 0.5
        }
    )
    
    segment: Optional[int] = field(
        default=None,
        metadata={
            SCHEMA_TITLE: "Segment Length",
            SCHEMA_DESC: "Split audio into segments of this many seconds for processing. None = process whole file."
        }
    )
    
    save_other_stems: bool = field(
        default=False,
        metadata={
            SCHEMA_TITLE: "Save Other Stems",
            SCHEMA_DESC: "Also save drums, bass, and other stems alongside vocals."
        }
    )
    
    output_format: str = field(
        default="wav",
        metadata={
            SCHEMA_TITLE: "Output Format",
            SCHEMA_DESC: "Format for output audio files.",
            SCHEMA_ENUM: ["wav", "flac", "mp3"]
        }
    )

## Plugin Class

In [ ]:
#| export
class DemucsProcessingPlugin(MediaProcessingPlugin):
    """Demucs v4 source separation plugin for vocals extraction."""
    
    config_class = DemucsPluginConfig
    
    def __init__(self):
        """Initialize the plugin."""
        self.logger = logging.getLogger(f"{__name__}.{type(self).__name__}")
        self.config: Optional[DemucsPluginConfig] = None
        self.storage: Optional[MediaProcessingStorage] = None
        self._data_dir: Optional[str] = None
        self._separator = None  # Lazy-loaded Demucs Separator
        self._loaded_model_name: Optional[str] = None  # Track which model is loaded
    
    # ── Properties ──────────────────────────────────────────────────
    
    @property
    def name(self) -> str:  # Plugin name identifier
        """Get the plugin name."""
        return "cjm-media-plugin-demucs"
    
    @property
    def version(self) -> str:  # Plugin version string
        """Get the plugin version."""
        from cjm_media_plugin_demucs import __version__
        return __version__
    
    @property
    def supported_media_types(self) -> List[str]:  # Supported media types
        """Get supported media types."""
        return ["audio"]
    
    # ── Lifecycle ────────────────────────────────────────────────────
    
    def initialize(self,
                   config: Optional[Any] = None,  # Configuration dict or None for defaults
                  ) -> None:
        """Initialize plugin with configuration."""
        self.config = dict_to_config(DemucsPluginConfig, config or {})
        
        from .meta import get_plugin_metadata
        metadata = get_plugin_metadata()
        db_path = metadata["db_path"]
        self._data_dir = str(Path(db_path).parent)
        
        self.storage = MediaProcessingStorage(db_path)
        self.logger.info(f"Initialized with model={self.config.model}, "
                         f"device={self.config.device}")
    
    def cleanup(self) -> None:
        """Clean up plugin resources."""
        self._unload_model()
        self.logger.info("Plugin cleaned up")
    
    def is_available(self) -> bool:  # Whether the plugin can run
        """Check if the plugin is available on this system."""
        try:
            import demucs  # noqa
            return True
        except ImportError:
            return False
    
    def get_config_schema(self) -> Dict[str, Any]:  # JSON Schema for UI forms
        """Return JSON Schema for the plugin configuration."""
        return dataclass_to_jsonschema(DemucsPluginConfig)
    
    def get_current_config(self) -> Dict[str, Any]:  # Current config as dict
        """Return the current configuration."""
        return config_to_dict(self.config) if self.config else {}
    
    # ── Model Management ────────────────────────────────────────────
    
    def _load_model(self) -> None:
        """Load or reload the Demucs Separator (lazy, cached)."""
        model_name = self.config.model
        
        # Already loaded with the same model
        if self._separator is not None and self._loaded_model_name == model_name:
            return
        
        # Different model requested — unload first
        if self._separator is not None:
            self.logger.info(f"Switching model: {self._loaded_model_name} -> {model_name}")
            self._unload_model()
        
        device = self.config.device
        if device == "auto":
            device = "cuda" if torch.cuda.is_available() else "cpu"
        
        self.logger.info(f"Loading Demucs model '{model_name}' on {device}...")
        from demucs.api import Separator
        self._separator = Separator(
            model=model_name,
            device=device,
            shifts=self.config.shifts,
            overlap=self.config.overlap,
            segment=self.config.segment,
        )
        self._loaded_model_name = model_name
        self.logger.info(f"Model loaded: samplerate={self._separator.samplerate}")
    
    def _unload_model(self) -> None:
        """Unload the Demucs model and free GPU memory."""
        if self._separator is not None:
            del self._separator
            self._separator = None
            self._loaded_model_name = None
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            self.logger.info("Model unloaded, CUDA cache cleared")
    
    # ── Job Storage ──────────────────────────────────────────────────
    
    def _store_job(self,
                   action: str,  # Action name
                   input_path: str,  # Source file path
                   output_path: str,  # Output file path
                   parameters: Optional[Dict[str, Any]] = None,  # Action parameters
                   metadata: Optional[Dict[str, Any]] = None,  # Additional metadata
                   input_hash: Optional[str] = None,  # Pre-computed input hash
                  ) -> str:  # Generated job_id
        """Hash input/output files and store a processing job record."""
        job_id = str(uuid.uuid4())
        if input_hash is None:
            input_hash = hash_file(input_path)
        output_hash = hash_file(output_path)
        self.storage.save(
            job_id=job_id, action=action,
            input_path=input_path, input_hash=input_hash,
            output_path=output_path, output_hash=output_hash,
            parameters=parameters, metadata=metadata
        )
        return job_id
    
    # ── Action Dispatch ──────────────────────────────────────────────
    
    def execute(self,
                action: str = "separate_vocals",  # Action to perform
                **kwargs
               ) -> Dict[str, Any]:  # Action result
        """Dispatch to the appropriate action handler."""
        if action == "get_info":
            return self.get_info(kwargs["file_path"]).to_dict()
        elif action == "separate_vocals":
            return self._separate_vocals(**kwargs)
        else:
            raise ValueError(f"Unknown action: {action}. Supported: 'get_info', 'separate_vocals'")
    
    # ── Interface Methods (not applicable) ───────────────────────────
    
    def get_info(self,
                 file_path: str,  # Path to audio file
                ) -> MediaMetadata:  # File metadata
        """Get basic audio file metadata via ffprobe."""
        from demucs.audio import AudioFile
        af = AudioFile(file_path)
        return MediaMetadata(
            path=file_path,
            format=Path(file_path).suffix.lstrip('.'),
            duration=af.duration,
            size_bytes=Path(file_path).stat().st_size,
            audio_streams=[{
                'channels': af.channels(),
                'sample_rate': af.samplerate(),
            }],
            video_streams=[],
        )
    
    def convert(self, input_path, output_format, **kwargs):
        """Not applicable for source separation."""
        raise ValueError("convert is not supported by the Demucs plugin. "
                         "Use 'separate_vocals' instead.")
    
    def extract_segment(self, input_path, start, end, output_path=None):
        """Not applicable for source separation."""
        raise ValueError("extract_segment is not supported by the Demucs plugin. "
                         "Use 'separate_vocals' instead.")
    
    # ── Core Action ──────────────────────────────────────────────────
    
    def _separate_vocals(self,
                         input_path: str,  # Path to audio file
                         output_dir: Optional[str] = None,  # Output directory (default: data_dir/vocals/)
                         output_format: Optional[str] = None,  # Output format override
                        ) -> Dict[str, Any]:  # Separation result
        """Extract vocals stem from an audio file."""
        fmt = output_format or self.config.output_format
        input_p = Path(input_path)
        
        # Determine output directory
        if output_dir is None:
            out_dir = Path(self._data_dir) / "vocals" / input_p.stem
        else:
            out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        
        # Load model (lazy, cached)
        self.report_progress(0.0, "Loading model...")
        self._load_model()
        
        # Run separation
        self.report_progress(0.1, "Separating audio...")
        origin, separated = self._separator.separate_audio_file(input_p)
        
        # Save vocals
        self.report_progress(0.8, "Saving vocals...")
        from demucs.audio import save_audio
        vocals_path = out_dir / f"vocals.{fmt}"
        save_audio(separated["vocals"], str(vocals_path),
                   samplerate=self._separator.samplerate)
        
        # Optionally save other stems
        other_stems_saved = []
        if self.config.save_other_stems:
            for stem_name, stem_tensor in separated.items():
                if stem_name == "vocals":
                    continue
                stem_path = out_dir / f"{stem_name}.{fmt}"
                save_audio(stem_tensor, str(stem_path),
                           samplerate=self._separator.samplerate)
                other_stems_saved.append(str(stem_path))
        
        # Hash and store job
        self.report_progress(0.9, "Storing job record...")
        job_id = self._store_job(
            action="separate_vocals",
            input_path=str(input_p),
            output_path=str(vocals_path),
            parameters={
                "model": self.config.model,
                "output_format": fmt,
                "shifts": self.config.shifts,
                "overlap": self.config.overlap,
            },
            metadata={
                "duration": float(separated["vocals"].shape[-1]) / self._separator.samplerate,
                "stems": list(separated.keys()),
                "device": str(self.config.device),
                "other_stems_saved": other_stems_saved,
            },
        )
        
        self.report_progress(1.0, "Complete")
        
        return {
            "job_id": job_id,
            "output_path": str(vocals_path),
            "input_path": str(input_p),
            "duration": float(separated["vocals"].shape[-1]) / self._separator.samplerate,
            "model": self.config.model,
            "stems_available": list(separated.keys()),
        }

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()